# Gradient Descent for Linear Regression

In this lab, implement gradient descent to fit a straight line to housing prices from one feature: house size in square metres.

## Learning goals

- Calculate the squared-error cost $J(w,b)$.
- Calculate gradients for the slope $w$ and intercept $b$.
- Use gradient descent to fit $f_{w,b}(x) = wx + b$.
- Observe how the learning rate affects training.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Housing-price dataset

This is the same seven-point example as the interactive visualizer. It uses seven observations from the public Ames Housing dataset: homes sold in Ames, Iowa, from 2006 to 2010. The original `GrLivArea` values are above-ground living area in ft²; they are converted here to m². `SalePrice` is expressed in thousands of US dollars.

Source: [Ames Housing dataset on OpenML](https://www.openml.org/d/42165). This lab deliberately uses only size as a feature, although real sale prices also depend on many other variables.

| House size (m²) | Price (thousand dollars) |
|---:|---:|
| 79.5 | 208.5 |
| 117.2 | 181.5 |
| 85.5 | 223.5 |
| 89.3 | 140.0 |
| 106.4 | 250.0 |
| 74.0 | 143.0 |
| 157.4 | 307.0 |

In [ ]:
# Ames Housing records 1-7: GrLivArea converted from ft² to m²; SalePrice in thousand USD.
x_train = np.array([79.5, 117.2, 85.5, 89.3, 106.4, 74.0, 157.4])
y_train = np.array([208.5, 181.5, 223.5, 140.0, 250.0, 143.0, 307.0])
m = x_train.shape[0]
print(f'Number of training examples: {m}')

In [ ]:
plt.scatter(x_train, y_train, color='tab:blue', label='Training data')
plt.xlabel('House size (m²)')
plt.ylabel('Price (thousand dollars)')
plt.title('Housing-price training data')
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## Cost function

The model is $f_{w,b}(x) = wx + b$. We measure its error with:

$$J(w,b) = \frac{1}{2m}\sum_{i=1}^{m}(f_{w,b}(x^{(i)}) - y^{(i)})^2$$

In [ ]:
def compute_cost(x, y, w, b):
    predictions = w * x + b
    return np.sum((predictions - y) ** 2) / (2 * x.shape[0])

print(f'Initial cost at w=0, b=0: {compute_cost(x_train, y_train, 0, 0):.2f}')

## Compute the gradients

For each step, calculate both gradients using the current values of $w$ and $b$:

$$\frac{\partial J}{\partial w} = \frac{1}{m}\sum_{i=1}^{m}(f_{w,b}(x^{(i)}) - y^{(i)})x^{(i)}$$

$$\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}(f_{w,b}(x^{(i)}) - y^{(i)})$$

In [ ]:
def compute_gradient(x, y, w, b):
    error = w * x + b - y
    dj_dw = np.sum(error * x) / x.shape[0]
    dj_db = np.sum(error) / x.shape[0]
    return dj_dw, dj_db

dj_dw, dj_db = compute_gradient(x_train, y_train, 0, 0)
print(f'Gradient at w=0, b=0: dj_dw={dj_dw:.2f}, dj_db={dj_db:.2f}')

## Implement gradient descent

Both parameters are updated simultaneously:

$$w := w - \alpha\frac{\partial J}{\partial w}, \quad b := b - \alpha\frac{\partial J}{\partial b}$$

Because house size is measured directly in m², use a small learning rate such as `0.00001`.

In [ ]:
def gradient_descent(x, y, w_start, b_start, alpha, iterations):
    w = w_start
    b = b_start
    cost_history = []
    parameter_history = []

    for _ in range(iterations):
        dj_dw, dj_db = compute_gradient(x, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        cost_history.append(compute_cost(x, y, w, b))
        parameter_history.append((w, b))

    return w, b, cost_history, parameter_history

In [ ]:
w_start = 0
b_start = 0
alpha = 0.00001
iterations = 100_000

w_final, b_final, cost_history, parameter_history = gradient_descent(
    x_train, y_train, w_start, b_start, alpha, iterations
)

print(f'Final w: {w_final:.3f} thousand dollars per m²')
print(f'Final b: {b_final:.3f} thousand dollars')
print(f'Final cost: {cost_history[-1]:.3f}')

In [ ]:
plt.plot(cost_history)
plt.xlabel('Iteration')
plt.ylabel('Cost J(w, b)')
plt.title('Cost decreases during gradient descent')
plt.grid(alpha=0.25)
plt.show()

In [ ]:
x_line = np.linspace(x_train.min(), x_train.max(), 100)
plt.scatter(x_train, y_train, color='tab:blue', label='Training data')
plt.plot(x_line, w_final * x_line + b_final, color='tab:red', label='Fitted line')
plt.xlabel('House size (m²)')
plt.ylabel('Price (thousand dollars)')
plt.title('Linear regression fitted with gradient descent')
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## Make a prediction

Use the fitted line to estimate a price for a 150 m² house. Then change the size and rerun the cell.

In [ ]:
house_size = 150
predicted_price = w_final * house_size + b_final
print(f'Predicted price for {house_size} m²: ${predicted_price * 1000:,.0f}')

## Experiment with the learning rate

Change `alpha` below. Try a smaller value such as `0.000001`, then a larger value such as `0.0001`. Compare the final cost and the cost plot.

In [ ]:
experiment_alpha = 0.0001
_, _, experiment_costs, _ = gradient_descent(
    x_train, y_train, 0, 0, experiment_alpha, 5_000
)

plt.plot(experiment_costs)
plt.xlabel('Iteration')
plt.ylabel('Cost J(w, b)')
plt.title(f'Learning-rate experiment: alpha={experiment_alpha}')
plt.grid(alpha=0.25)
plt.show()
print(f'Final cost: {experiment_costs[-1]:.3f}')

## Summary

You implemented univariate linear regression with gradient descent on the same housing-price dataset as the interactive demo. The feature is house size in m², and gradient descent learned the line parameters $w$ and $b$ by reducing the squared-error cost.